# WEEK 4 - HỆ THỐNG LIVE DEMO TÍCH HỢP (xAI ENGINE)
========================================================================
**Luồng xử lý tích hợp:** 
1. **Track C (Consensus Persona):** Phân cụm hành vi khách hàng sử dụng mô hình đồng thuận.
2. **Track A (Fraud & Anomaly):** Đánh giá rủi ro giao dịch bất thường & chỉ ra lý do cụ thể.
3. **Track B (NBFO):** Đề xuất sản phẩm tối ưu (Thẻ/Tiết kiệm/Vay) dựa trên Percentile-rank, áp dụng chính sách chặn rủi ro an toàn.

In [1]:
# Khởi tạo môi trường và nạp thư viện
import sys
import os
import pandas as pd
import numpy as np

# Đưa thư mục src vào hệ thống để import native các hàm xử lý
sys.path.insert(0, os.path.abspath('..'))
from src.week4_demo_lookup import print_customer, choose_customer

# Thiết lập đường dẫn dữ liệu demo
demo_csv_path = '../outputs/demo/week4_demo_customers.csv'
if not os.path.exists(demo_csv_path):
    raise FileNotFoundError(f"Không tìm thấy file dữ liệu demo tại {demo_csv_path}. Vui lòng kiểm tra lại thư mục outputs.")

df_demo = pd.read_csv(demo_csv_path)
print(f"Đã nạp thành công bộ dữ liệu demo gồm {len(df_demo)} khách hàng đại diện thuộc tập holdout test.")

Đã nạp thành công bộ dữ liệu demo gồm 49 khách hàng đại diện thuộc tập holdout test.


## PHẦN 1: TỔNG QUAN PHÂN BỔ CÁC TRƯỜNG HỢP DEMO
-----------------------------------------------------------------------

In [2]:
summary = df_demo["demo_case"].value_counts().rename_axis("Kịch bản Demo").reset_index(name="Số lượng khách")
summary

,Kịch bản Demo,Số lượng khách
0,random_holdout_customer,20
1,high_fraud_risk,5
2,low_confidence_no_offer,3
3,top_card_offer,3
4,top_loan_offer,3
5,top_td_offer,3
6,wealth_Bottom90,3
7,wealth_P90_P95,3
8,wealth_P95_P99,3
9,wealth_P99_top1,3


## PHẦN 2: CÁC CASE ĐỀ XUẤT SẢN PHẨM CHÍNH XÁC CAO (TOP PROPENSITY MATCH)
------------------------------------------------------------------------
### Case 2.1: Đề xuất sản phẩm Vay vốn thành công (Loan Offer)
*   **Mã khách hàng:** `751560`
*   **Thuyết minh:** Điểm propensity vay đạt `0.9854` (hạng phân vị `1.0000`). Đối chiếu nhãn thực tế tháng sau của tập holdout: `label_loan = 1` (Khách thực sự vay thật, dự đoán đúng 100%).

In [3]:
row = choose_customer(df_demo, customer=751560, case=None)
print_customer(row)

Customer: 751560
Demo case: top_loan_offer
----------------------------------------------------------------------------------------
1) Persona
   Cluster: 4
   Persona confidence: 19.57%
   Wealth segment: Bottom90
   Age: 26.9
----------------------------------------------------------------------------------------
2) Fraud risk
   Fraud score: 0.0000
   Fraud flag: 0.0000
   Reason: stable baseline profile
----------------------------------------------------------------------------------------
3) Next Best Offer
   Best offer: loan
   Top offers: loan
   Max rank: 1.0000
   Offer reason: Choose LOAN because it has the highest rank (rank=1.0000, score=0.9854).
----------------------------------------------------------------------------------------
4) Holdout labels for checking
   loan=1, td=0, card=0
----------------------------------------------------------------------------------------
5) Main profile signals
   has_ebanking: 1
   act_count: 8.0000
   login_count: 1.0000
   trans_co

### Case 2.2: Đề xuất sản phẩm Thẻ tín dụng thành công (Card Offer)
*   **Mã khách hàng:** `44703`
*   **Thuyết minh:** Điểm propensity thẻ đạt `0.9407` (hạng phân vị `1.0000`). Nhãn kiểm chứng tháng sau đạt **`label_card = 1`** (Khách mở thẻ thật, dự đoán đúng 100%).

In [ ]:
row = choose_customer(df_demo, customer=44703, case=None)
print_customer(row)

### Case 2.3: Đề xuất sản phẩm Tiết kiệm thành công (Term Deposit Offer)
*   **Mã khách hàng:** `518080`
*   **Thuyết minh:** Điểm propensity tiết kiệm đạt `0.9694` (hạng phân vị `1.0000`). Nhãn kiểm chứng tháng sau đạt **`label_td = 1`** (Khách gửi tiết kiệm thật, dự đoán đúng 100%).

In [ ]:
row = choose_customer(df_demo, customer=518080, case=None)
print_customer(row)

## PHẦN 3: CÁC CASE CHẶN RỦI RO GIAN LẬN & PHÒNG CHỐNG RỬA TIỀN (RISK-AWARE POLICY)
------------------------------------------------------------------------
### Case 3.1: Khách hàng siêu VIP có dòng tiền ra ngoài bất thường
*   **Mã khách hàng:** `509963`
*   **Thuyết minh:** Khách hàng thuộc top 1% siêu giàu (`P99_top1`) có hơn 4.3 tỷ VND tiết kiệm. Mặc dù propensity thẻ rất cao, nhưng Track A phát hiện rủi ro gian lận vượt ngưỡng (`Fraud flag = 1.0`) do chuyển tiền ngoài ngân hàng chiếm 68.97%. Hệ thống tự động chặn ưu đãi và chuyển sang trạng thái tạm giữ rủi ro (`risk_hold`).

In [ ]:
row = choose_customer(df_demo, customer=509963, case=None)
print_customer(row)

### Case 3.2: Khách hàng hoạt động đêm và chuyển tiền ngoài cực đoan
*   **Mã khách hàng:** `413763`
*   **Thuyết minh:** Hệ thống phát hiện khách hàng có 40% số giao dịch diễn ra ban đêm và 100% dòng tiền chuyển ra ngoài hệ thống ngân hàng. Trạng thái cảnh báo rủi ro được kích hoạt tức thì.

In [ ]:
row = choose_customer(df_demo, customer=413763, case=None)
print_customer(row)

## PHẦN 4: CASE KHÔNG ĐỀ XUẤT ƯU ĐÃI ĐỂ TIẾT KIỆM CHI PHÍ MARKETING (NO OFFER)
------------------------------------------------------------------------
### Case 4.1: Khách hàng trẻ tuổi, không có tương tác số
*   **Mã khách hàng:** `855396`
*   **Thuyết minh:** Khách hàng mới 18 tuổi, số dư cực thấp (~114k CA), hoàn toàn không có tương tác số (đăng nhập/hoạt động bằng 0). Mô hình từ chối đề xuất (`no_offer`) do cả 3 phân vị propensity đều tiệm cận về 0.

In [ ]:
row = choose_customer(df_demo, customer=855396, case=None)
print_customer(row)

## PHẦN 5: CÁC CASE KHÁCH HÀNG VIP AN TOÀN (WEALTHY VIP SEGMETN)
------------------------------------------------------------------------
### Case 5.1: Khách hàng siêu VIP gửi tiết kiệm lớn, rủi ro thấp
*   **Mã khách hàng:** `732351`
*   **Thuyết minh:** Khách hàng thuộc phân vị tài sản P99 cao nhất (có 800 triệu VND gửi tiết kiệm). Rủi ro bằng 0. Đề xuất tối ưu đưa ra là sản phẩm vay vốn để tiếp tục cross-sell.

In [ ]:
row = choose_customer(df_demo, customer=732351, case=None)
print_customer(row)

## PHẦN 6: TRA CỨU TỰ DO THEO ID KHÁCH HÀNG (DÀNH CHO GIÁO VIÊN HỎI)
------------------------------------------------------------------------
Thầy cô có thể chọn một ID bất kỳ trong danh sách dưới đây để nhập vào ô tra cứu trực tiếp:

*   **Nhóm VIP an toàn:** `865283`, `69788`, `455413`
*   **Nhóm Rủi ro đáng ngờ:** `444248`, `609360`, `747140`
*   **Nhóm Đề xuất chính xác:** `482901`, `772432`, `377287`, `310093`, `685170`
*   **Nhóm không hoạt động:** `5234`, `106640`

In [ ]:
# NHẬP MÃ KHÁCH HÀNG BẤT KỲ VÀO ĐÂY ĐỂ TRA CỨU
ID_TRA_CUU = 865283 

try:
    selected_row = choose_customer(df_demo, customer=ID_TRA_CUU, case=None)
    print_customer(selected_row)
except Exception as e:
    print(f"Lỗi: {e}. Vui lòng nhập đúng mã ID nằm trong danh sách demo khách hàng holdout.")

## PHẦN 7: BỔ SUNG NGƯỜI DÙNG DỰ TRÊN ID HOẶC CHỌN NGẪU NHIÊN (DYNAMIC xAI)
------------------------------------------------------------------------
**Thuyết minh trước hội đồng:**
*"Kính thưa thầy/cô, để chứng minh tính tự động hóa hoàn toàn và khả năng mở rộng của hệ thống, phần này cho phép nạp thêm bất kỳ khách hàng mới nào từ tập kiểm thử holdout test vào tập demo. Hệ thống sẽ tự động trích xuất đặc trưng, chấm điểm rủi ro, chạy decision layer đề xuất sản phẩm và cập nhật trực tiếp vào cơ sở dữ liệu demo chỉ trong vài giây."*

In [ ]:
# Khởi tạo hàm bổ sung khách hàng động
def add_and_show_customer(customer_id=None, add_random=False, random_seed=None):
    import pandas as pd
    import numpy as np
    import os
    from src.week4_demo_lookup import print_customer
    
    # Thiet lap duong dan den cac file trong thu muc Baitaplon_Final
    demo_path = '../outputs/demo/week4_demo_customers.csv'
    decisions_path = '../outputs/tables/b_risk_aware_decision_customer_offers.csv'
    features_path = '../outputs/processed/persona_zoo/kmeans/track_b_features.parquet'
    
    df_demo = pd.read_csv(demo_path)
    df_dec = pd.read_csv(decisions_path)
    
    if add_random:
        # Loc cac ID thuoc tap holdout chua nam trong danh sach demo hien tai
        available_ids = df_dec[~df_dec['CUSTOMER_NUMBER'].isin(df_demo['CUSTOMER_NUMBER'])]['CUSTOMER_NUMBER'].tolist()
        if not available_ids:
            print("Khong con khach hang moi nao trong tap holdout de bo sung.")
            return
        rng = np.random.default_rng(random_seed)
        customer_id = int(rng.choice(available_ids))
        print(f"[RANDOM SELECTION] Da chon ngau nhien khach hang moi: {customer_id}")
    else:
        if customer_id is None:
            print("Vui long nhap CUSTOMER_NUMBER de bat dau.")
            return
        customer_id = int(customer_id)
        if customer_id in df_demo['CUSTOMER_NUMBER'].values:
            print(f"Khach hang {customer_id} da co san trong bo du lieu Demo. Duoi day la ket qua tra cuu:")
            row = df_demo[df_demo['CUSTOMER_NUMBER'] == customer_id].iloc[0]
            print_customer(row)
            return
        if customer_id not in df_dec['CUSTOMER_NUMBER'].values:
            print(f"Mã khach hang {customer_id} khong ton tai trong file cham diem rui ro/de xuat (hoac khong thuoc tap test).")
            return
            
    # Load dac trung tu file parquet de lay Main Profile Signals
    X = pd.read_parquet(features_path)
    if customer_id not in X.index:
        print(f"Khong tim thay dac trung cua khach hang {customer_id} trong du lieu parquet.")
        return
        
    dec_row = df_dec[df_dec['CUSTOMER_NUMBER'] == customer_id].iloc[0]
    feat_row = X.loc[customer_id]
    
    # Tinh toan Wealth Segment dong thoi gian thuc
    ca = feat_row.get("avg_ca_balance", 0)
    td = feat_row.get("avg_td_balance", 0)
    credit = feat_row.get("max_credit_card", 0)
    wealth_proxy = np.log1p(np.maximum(ca, 0) + np.maximum(td, 0) + 0.2 * np.maximum(credit, 0))
    
    all_wealth = np.log1p(np.maximum(X["avg_ca_balance"], 0) + np.maximum(X["avg_td_balance"], 0) + 0.2 * np.maximum(X.get("max_credit_card", 0), 0))
    q90, q95, q99 = all_wealth.quantile([0.90, 0.95, 0.99]).to_numpy()
    
    wealth_segment = "Bottom90"
    if wealth_proxy >= q99:
        wealth_segment = "P99_top1"
    elif wealth_proxy >= q95:
        wealth_segment = "P95_P99"
    elif wealth_proxy >= q90:
        wealth_segment = "P90_P95"
        
    # Sinh ly do xAI tu dong
    reasons = []
    if wealth_segment in {"P99_top1", "P95_P99"}:
        reasons.append("high-value wealth segment")
    if feat_row.get("act_count", 0) > 0 and (feat_row.get("login_count", 0)/max(feat_row.get("act_count", 1), 1)) > 0.5:
        reasons.append("active digital user")
    if feat_row.get("category_diversity", 0) >= 3:
        reasons.append("diverse transaction behavior")
    if feat_row.get("outside_bank_ratio", 0) > 0.3:
        reasons.append("high outside-bank transfer ratio")
    if feat_row.get("night_ratio", 0) > 0.2:
        reasons.append("unusual night transaction ratio")
    if not reasons:
        reasons.append("stable baseline profile")
    demo_reason = "; ".join(reasons[:3])
    
    # Xay dung dong moi hop le
    new_row_dict = {
        "CUSTOMER_NUMBER": customer_id,
        "demo_case": "dynamic_added_customer" if not add_random else "dynamic_added_random_customer",
        "best_offer": dec_row.get("best_offer", "no_offer"),
        "top3_offers": dec_row.get("top3_offers", "N/A"),
        "max_rank": dec_row.get("max_rank", np.nan),
        "persona_cluster": int(feat_row.get("persona_cluster", 0)),
        "persona_confidence": feat_row.get("persona_confidence", np.nan),
        "wealth_segment": wealth_segment,
        "wealth_proxy": wealth_proxy,
        "label_loan": int(dec_row.get("y_loan", 0)),
        "label_td": int(dec_row.get("y_td", 0)),
        "label_card": int(dec_row.get("y_card", 0)),
        "score_loan": dec_row.get("score_loan", np.nan),
        "score_td": dec_row.get("score_td", np.nan),
        "score_card": dec_row.get("score_card", np.nan),
        "rank_loan": dec_row.get("rank_loan", np.nan),
        "rank_td": dec_row.get("rank_td", np.nan),
        "rank_card": dec_row.get("rank_card", np.nan),
        "fraud_score": dec_row.get("fraud_score", np.nan),
        "fraud_pred": dec_row.get("fraud_pred", np.nan),
        "demo_reason": demo_reason,
        "age": feat_row.get("age", np.nan),
        "has_ebanking": int(feat_row.get("has_ebanking", 0)),
        "act_count": feat_row.get("act_count", 0),
        "login_count": feat_row.get("login_count", 0),
        "trans_count": feat_row.get("trans_count", 0),
        "category_diversity": feat_row.get("category_diversity", 0),
        "avg_ca_balance": feat_row.get("avg_ca_balance", 0),
        "avg_td_balance": feat_row.get("avg_td_balance", 0),
        "product_count": feat_row.get("product_count", 0),
        "outside_bank_ratio": feat_row.get("outside_bank_ratio", 0),
        "night_ratio": feat_row.get("night_ratio", 0)
    }
    
    new_df_row = pd.DataFrame([new_row_dict])
    
    # Append va ghi de lai vao CSV demo
    df_new_demo = pd.concat([df_demo, new_df_row], ignore_index=True)
    df_new_demo.to_csv(demo_path, index=False)
    
    print(f"Thanh cong! Da cap nhat va bo sung khach hang {customer_id} vao tap Demo!")
    print("=" * 88)
    print_customer(new_df_row.iloc[0])

print("Da khoi tao thanh cong ham add_and_show_customer!")

### Phần 7.1: Bổ sung khách hàng theo ID bất kỳ do Giáo viên yêu cầu
Nhập một mã `CUSTOMER_NUMBER` hợp lệ trong cơ sở dữ liệu test để chạy nạp động.

In [ ]:
# TUY CHON: Thay doi ma khach hang bat ky duoi day de test
CUSTOMER_ID_DE_ADD = 609360 

add_and_show_customer(customer_id=CUSTOMER_ID_DE_ADD)

### Phần 7.2: Chọn ngẫu nhiên một khách hàng mới tinh từ tập kiểm thử
Chạy cell dưới đây để chọn ngẫu nhiên một khách hàng hoàn toàn mới chưa từng xuất hiện trong tập demo để đánh giá.

In [ ]:
# Chay de lay ngau nhien mot khach hang va tu dong nap vao tap demo
add_and_show_customer(add_random=True)